In [ ]:
import pandas as pd
import numpy as np

# 1. Load raw dataset with semicolon delimiter
df_raw = pd.read_csv('Country Data Comparison.csv', sep=';')

print("Raw Data Shape")
print(df_raw.shape)

Raw Data Shape
(264, 45)


In [ ]:
# Drop metadata row and trim column headers
df_cleaned = df_raw.iloc[1:].reset_index(drop=True)
df_cleaned.columns = df_cleaned.columns.str.strip()

#Conversion all columns (except Country) to numeric floats
num_cols = [col for col in df_cleaned.columns if col != 'Country']

for col in num_cols:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

print(df_cleaned.shape)

(263, 45)


In [ ]:
print(f"\nDuplicate Countries Found: {df_cleaned['Country'].duplicated().sum()}")


Duplicate Countries Found: 0


In [ ]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 45 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Country                                         263 non-null    object 
 1   Area(sq km)                                     263 non-null    int64  
 2   Birth rate(births/1000 population)              225 non-null    float64
 3   Current account balance                         149 non-null    float64
 4   Death rate(deaths/1000 population)              225 non-null    float64
 5   Debt - external                                 201 non-null    float64
 6   Electricity - consumption(kWh)                  215 non-null    float64
 7   Electricity - production(kWh)                   213 non-null    float64
 8   Exports                                         224 non-null    float64
 9   GDP                                        

In [ ]:
df_cleaned.describe()

,Area(sq km),Birth rate(births/1000 population),Current account balance,Death rate(deaths/1000 population),Debt - external,Electricity - consumption(kWh),Electricity - production(kWh),Exports,GDP,GDP - per capita,...,Oil - production(bbl/day),Oil - proved reserves(bbl),Population,Public debt(% of GDP),Railways(km),Reserves of foreign exchange & gold,Telephones - main lines in use,Telephones - mobile cellular,Total fertility rate(children born/woman),Unemployment rate(%)
count,2.630000e+02,225.000000,1.490000e+02,225.000000,2.010000e+02,2.150000e+02,2.130000e+02,2.240000e+02,2.300000e+02,230.000000,...,2.120000e+02,9.700000e+01,2.380000e+02,108.000000,134.000000,1.470000e+02,2.320000e+02,2.280000e+02,225.000000,192.000000
mean,5.849875e+05,22.146667,-8.473897e+08,9.374267,6.317084e+10,8.070919e+10,8.849167e+10,4.432637e+10,2.925614e+11,10552.760870,...,3.710435e+05,1.416887e+10,2.900494e+07,55.896296,10217.246269,2.599825e+10,5.998770e+06,7.331346e+06,2.937778,15.254687
std,1.881416e+06,11.126969,5.633953e+10,4.969136,3.588986e+11,3.434223e+11,3.706046e+11,1.334267e+11,1.252331e+12,11104.610351,...,1.190319e+06,3.923463e+10,1.175456e+08,36.954000,29857.440213,8.023959e+10,2.717322e+07,3.075963e+07,1.556919,15.113801
min,0.000000e+00,7.230000,-6.465000e+11,2.300000,0.000000e+00,1.674000e+06,1.800000e+06,9.800000e+04,1.500000e+06,400.000000,...,0.000000e+00,0.000000e+00,4.600000e+01,2.100000,6.000000,2.978000e+07,0.000000e+00,0.000000e+00,0.910000,0.000000
25%,5.130000e+02,12.570000,-5.908000e+08,6.070000,6.680000e+08,4.529000e+08,4.108000e+08,2.365000e+08,2.522500e+09,2000.000000,...,0.000000e+00,8.000000e+07,2.837770e+05,31.475000,801.250000,7.851000e+08,3.365000e+04,4.137500e+04,1.750000,5.575000
50%,4.884500e+04,18.780000,-8.240000e+07,8.000000,4.069000e+09,4.450000e+09,4.873000e+09,2.058000e+09,1.685500e+10,5900.000000,...,0.000000e+00,4.088000e+08,4.440570e+06,50.750000,2585.500000,3.034000e+09,2.313500e+05,3.500500e+05,2.370000,10.500000
75%,3.339475e+05,30.010000,1.400000e+09,11.260000,1.707000e+10,3.320500e+10,3.551000e+10,1.624250e+10,9.183250e+10,16700.000000,...,7.214500e+04,4.408000e+09,1.628023e+07,69.675000,5195.000000,1.625500e+10,1.828550e+06,2.605450e+06,3.930000,19.625000
max,1.707520e+07,48.300000,1.702000e+11,29.360000,4.710000e+12,3.660000e+12,3.839000e+12,1.109000e+12,1.175000e+13,58900.000000,...,9.021000e+06,2.617000e+11,1.306314e+09,228.300000,228464.000000,6.646000e+11,2.630000e+08,3.146447e+08,6.840000,90.000000


In [ ]:
# 4. Handle Missing Values (Imputation)
df_imputed = df_cleaned.copy()

zero_fill_cols = [
    'Natural gas - consumption(cu m)', 'Natural gas - exports(cu m)',
    'Natural gas - imports(cu m)', 'Natural gas - production(cu m)',
    'Natural gas - proved reserves(cu m)', 'Oil - consumption(bbl/day)',
    'Oil - exports(bbl/day)', 'Oil - imports(bbl/day)',
    'Oil - production(bbl/day)', 'Oil - proved reserves(bbl)',
    'Railways(km)', 'HIV/AIDS - deaths', 'HIV/AIDS - people living with HIV/AIDS'
]

# Median Fill
median_fill_cols = [c for c in num_cols if c not in zero_fill_cols]

# Apply Zero Fill
df_imputed[zero_fill_cols] = df_imputed[zero_fill_cols].fillna(0)

# Apply Median Fill
for col in median_fill_cols:
    if col in df_imputed.columns:
        df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

print("\n--- IMPUTATION COMPLETE ---")
print("Total missing values remaining:", df_imputed.isnull().sum().sum())

# Save finalized dataset
df_imputed.to_csv('Cleaned_Imputed_Country_Data.csv', index=False)


--- IMPUTATION COMPLETE ---
Total missing values remaining: 0


In [ ]:
df_raw.shape

(264, 45)